# 🤖 MiniGPT — Transformer do Zero com PyTorch
### *Treinado em Dom Casmurro · Machado de Assis*

---

Neste notebook vamos construir um **Transformer Decoder-Only** (estilo GPT) completamente do zero usando PyTorch, e treiná-lo para gerar texto ao estilo de Machado de Assis.

## 🎯 Objetivos

1. Entender e implementar um **tokenizador char-level**
2. Montar um **Dataset** com janela deslizante para modelagem de linguagem
3. Implementar **Causal Self-Attention** (atenção mascarada)
4. Empilhar blocos Transformer para formar um **modelo GPT completo**
5. Treinar e **gerar texto** com sampling por temperatura e top-k

## 🏗️ Arquitetura: Decoder-Only Transformer (GPT-style)

```
                  "Capitu era..."
                        │
              ┌─────────▼──────────┐
              │   Tokenizador       │  char → inteiro
              │   CharTokenizer     │
              └─────────┬──────────┘
                        │  [10, 3, 55, 24, ...]
              ┌─────────▼──────────┐
              │  Token Embedding   │  (B, T) → (B, T, d_model)
              │+ Pos. Embedding    │
              └─────────┬──────────┘
                        │
              ┌─────────▼──────────┐
              │  Transformer Block │  × N camadas
              │  ┌──────────────┐  │
              │  │ LayerNorm    │  │
              │  │ Causal Self- │  │  cada token olha só
              │  │  Attention   │  │  para o passado ◀
              │  │ + Residual   │  │
              │  ├──────────────┤  │
              │  │ LayerNorm    │  │
              │  │ FeedForward  │  │  transforma cada
              │  │   (MLP 4×)   │  │  token isoladamente
              │  │ + Residual   │  │
              │  └──────────────┘  │
              └─────────┬──────────┘
                        │
              ┌─────────▼──────────┐
              │  LayerNorm + Head  │  (B, T, d_model) → (B, T, vocab)
              └─────────┬──────────┘
                        │
              Logits → Próximo token
```

> **Por que Decoder-Only?** Modelos de *geração* de texto só precisam "olhar para o passado" — cada token prediz o próximo. O Encoder só é necessário quando processamos uma sequência inteira *antes* de produzir outra (ex: tradução).

---
**⚙️ Configuração recomendada:** Google Colab com GPU T4 (Runtime → Change runtime type → T4 GPU)


In [ ]:
import torch, subprocess

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"SM       : {torch.cuda.get_device_capability(0)}")

# Teste mínimo de CUDA
try:
    x = torch.zeros(4, device="cuda")
    print(f"CUDA OK  : {x}")
except Exception as e:
    print(f"CUDA FAIL: {e}")

In [ ]:
# ── QUICK FIX — cole e execute esta célula ANTES de todas as outras ───────────
import torch, os

# 1. Desabilitar compile globalmente
os.environ["TORCHDYNAMO_DISABLE"] = "1"
torch._dynamo.config.suppress_errors = True

# 2. Se model já existe e está compilado, desempacotar
if "model" in dir():
    if hasattr(model, "_orig_mod"):
        model = model._orig_mod
        model.eval()
        print("✅ Model desempacotado (removido wrapper do compile)")

# 3. Confirmar
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Compilado? : {hasattr(model, '_orig_mod') if 'model' in dir() else 'model não existe ainda'}")

In [ ]:
# Verificar disponibilidade de GPU
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponível : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsando dispositivo: {device}")


In [ ]:
# ── Detecção automática de ambiente ──────────────────────────────────────────
import os, sys

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE:
    OUTPUT_DIR  = "/kaggle/working"
    USAR_DRIVE  = False
    # torch.compile usa o backend "inductor" que tenta criar kernels CUDA em
    # tempo de execução. Em diversas instâncias do Kaggle o kernel gerado é
    # incompatível com a GPU alocada (cudaErrorNoKernelImageForDevice), o que
    # corrompe o contexto CUDA e derruba todas as operações seguintes.
    # Desabilitamos preventivamente — AMP já entrega ~2× de ganho sozinho.
    os.environ["TORCHDYNAMO_DISABLE"] = "1"   # bloqueia qualquer tentativa
    print("✅ Ambiente detectado: Kaggle")
    print(f"   Modelos salvos em : {OUTPUT_DIR}")
    print(f"   torch.compile     : DESABILITADO (incompatível com esta GPU)")

elif IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        OUTPUT_DIR = "/content/drive/MyDrive/MiniGPT_DomCasmurro"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        USAR_DRIVE = True
        print("✅ Ambiente detectado: Google Colab")
        print(f"   Modelos salvos em : {OUTPUT_DIR}")
    except Exception as e:
        OUTPUT_DIR = "/content"
        USAR_DRIVE = False
        print(f"⚠️  Drive não montado ({e}) — salvando em {OUTPUT_DIR}")

else:
    OUTPUT_DIR = "."
    USAR_DRIVE = False
    print("✅ Ambiente detectado: Local")
    print(f"   Modelos salvos em : {os.path.abspath(OUTPUT_DIR)}")

print(f"\n   IS_KAGGLE = {IS_KAGGLE}")
print(f"   IS_COLAB  = {IS_COLAB}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler   # ← Mixed Precision (AMP)
from typing import Optional
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import requests
import re
import math
import time
from collections import Counter

# Reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports concluídos")


---
## 📚 Módulo 1 — Corpus: Dom Casmurro

Vamos baixar diretamente do **Project Gutenberg** a versão em português de *Dom Casmurro* (1899), de Machado de Assis.

### Por que Machado de Assis?
- 📜 **Domínio público** — sem restrições de direitos autorais
- 🖊️ **Estilo marcante** — voz narrativa única, fácil de reconhecer no output
- 📖 **Volume adequado** — ~320.000 caracteres, suficiente para um modelo didático

### Pipeline de dados

```
Project Gutenberg  ──►  Download  ──►  Limpeza  ──►  Corpus (texto puro)
```


In [ ]:
def download_dom_casmurro() -> str:
    """Baixa Dom Casmurro do Project Gutenberg e limpa o texto."""
    urls = [
        "https://www.gutenberg.org/cache/epub/55752/pg55752.txt",
        "https://www.gutenberg.org/files/55752/55752-0.txt",
    ]

    text = None
    for url in urls:
        try:
            print(f"Tentando: {url}")
            r = requests.get(url, timeout=30)
            r.encoding = "utf-8"
            text = r.text
            print(f"✅ Download concluído ({len(text):,} chars brutos)")
            break
        except Exception as e:
            print(f"  ✗ Falhou: {e}")

    if text is None:
        raise RuntimeError(
            "Não foi possível baixar. Faça upload manual do arquivo .txt"
        )

    # ── Remover cabeçalho e rodapé do Gutenberg ──────────────────────────
    marcadores_inicio = ["CAPÍTULO I", "Capítulo I", "CAPITULO I"]
    marcador_fim      = "*** END OF THE PROJECT GUTENBERG"

    for m in marcadores_inicio:
        idx = text.find(m)
        if idx != -1:
            text = text[idx:]
            break

    idx_fim = text.find(marcador_fim)
    if idx_fim != -1:
        text = text[:idx_fim]

    # ── Limpeza básica ────────────────────────────────────────────────────
    text = re.sub(r"\r\n", "\n", text)        # normaliza quebras de linha
    text = re.sub(r"\n{3,}", "\n\n", text)    # remove linhas em branco excessivas
    text = text.strip()

    print(f"\n📊 Estatísticas do corpus limpo:")
    print(f"   Caracteres : {len(text):>10,}")
    print(f"   Palavras   : {len(text.split()):>10,}")
    print(f"   Linhas     : {len(text.splitlines()):>10,}")

    return text


corpus = download_dom_casmurro()

print("\n--- Primeiros 500 caracteres ---")
print(corpus[:500])


In [ ]:
# ── Análise exploratória do corpus ──────────────────────────────────────────
chars_unicos = sorted(set(corpus))

print("=" * 55)
print("  ESTATÍSTICAS DO CORPUS")
print("=" * 55)
print(f"  Vocabulário (chars únicos) : {len(chars_unicos)}")
print(f"  Charset completo           :")
print(f"  {''.join(chars_unicos)}")

# Top 15 mais frequentes
print("\nTop 15 caracteres mais frequentes:")
char_counts = Counter(corpus)
for ch, cnt in char_counts.most_common(15):
    bar  = "█" * (cnt // 2000)
    disp = repr(ch) if ch in ("\n", "\t", " ") else ch
    print(f"  {disp!r:6s}: {cnt:7,}  {bar}")

# Comprimento médio das palavras
palavras = corpus.split()
lens = [len(p) for p in palavras]
print(f"\nComprimento médio de palavras : {np.mean(lens):.1f} chars")
print(f"Palavra mais longa            : '{max(palavras, key=len)}'")

# Distribuição de comprimentos
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(lens, bins=range(1, 25), color="steelblue", edgecolor="white", alpha=0.85)
ax.set_title("Distribuição do comprimento das palavras — Dom Casmurro")
ax.set_xlabel("Comprimento (chars)")
ax.set_ylabel("Frequência")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


---
## 🔤 Módulo 2 — Tokenizador Char-Level

### O que é tokenização?

Redes neurais só entendem números. A tokenização converte texto em **sequências de inteiros** que o modelo processa.

### Comparação de abordagens

| Abordagem | Vocab | Pro | Contra |
|-----------|-------|-----|--------|
| **Char-level** ← usamos | ~80 | Simples, sem OOV¹ | Sequências longas |
| Word-level | ~50k | Intuitivo | Palavras novas = problema |
| BPE / WordPiece | ~32k | Balanceado | Complexo de implementar |

> ¹ **OOV** = *Out-of-Vocabulary* — palavra que o modelo nunca viu no treino.

### Como funciona nosso tokenizador

Coletamos todos os **caracteres únicos** do corpus e atribuímos um índice a cada um:

```
char → índice       índice → char
─────────────       ─────────────
'a'  →  3           3  → 'a'
'b'  →  4           4  → 'b'
'C'  → 12          12  → 'C'
'ã'  → 67          67  → 'ã'
...
```

A codificação do texto "Dom" ficaria `[23, 55, 49]` — apenas três inteiros.


In [ ]:
class CharTokenizer:
    """
    Tokenizador bidirecional no nível de caractere.

    Atributos:
        vocab_size   : número de caracteres únicos no corpus
        char_to_idx  : dict  char → int
        idx_to_char  : dict  int  → char
    """

    def __init__(self, text: str):
        self.chars       = sorted(set(text))
        self.vocab_size  = len(self.chars)
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}
        print(f"✅ Tokenizador criado  —  vocab_size = {self.vocab_size}")

    def encode(self, text: str) -> list:
        """String → lista de inteiros."""
        return [self.char_to_idx[ch] for ch in text]

    def decode(self, indices) -> str:
        """Lista de inteiros → string."""
        return "".join(self.idx_to_char[i] for i in indices)

    def __len__(self):
        return self.vocab_size


# ── Criar tokenizador e testar ───────────────────────────────────────────────
tokenizer = CharTokenizer(corpus)

trecho = "Capitu era formosa, de olhos de ressaca."
enc    = tokenizer.encode(trecho)
dec    = tokenizer.decode(enc)

print(f"\nOriginal   : {trecho}")
print(f"Codificado : {enc}")
print(f"Decodificado: {dec}")
print(f"Round-trip : {'✅ OK' if trecho == dec else '❌ ERRO'}")

# Visualizar mapeamento
print("\nPrimeiros 20 mapeamentos do vocabulário:")
for i, ch in enumerate(tokenizer.chars[:20]):
    print(f"  {i:3d} → {repr(ch)}")


In [ ]:
# ── Tokenizar o corpus inteiro ───────────────────────────────────────────────
print("Tokenizando corpus completo...")
data = torch.tensor(tokenizer.encode(corpus), dtype=torch.long)
print(f"Shape do tensor  : {data.shape}")
print(f"Primeiros tokens : {data[:30].tolist()}")
print(f"Decodificados    : {repr(tokenizer.decode(data[:30].tolist()))}")

# ── Divisão treino / validação (90% / 10%) ───────────────────────────────────
n     = len(data)
split = int(0.9 * n)
train_data = data[:split]
val_data   = data[split:]

print(f"\nDivisão do dataset:")
print(f"  Total      : {n:>10,} tokens")
print(f"  Treino     : {len(train_data):>10,} ({100*len(train_data)/n:.0f}%)")
print(f"  Validação  : {len(val_data):>10,} ({100*len(val_data)/n:.0f}%)")


---
## 📦 Módulo 3 — Dataset com Janela Deslizante

### Modelagem de Linguagem Autoregressiva

O objetivo é simples: **dado um contexto de T caracteres, prever o próximo caractere**.

### Técnica: Teacher Forcing

Durante o treino, o modelo recebe a sequência **correta** como entrada (não suas próprias previsões). Assim ele aprende a prever *todos* os próximos tokens de uma vez em paralelo:

```
Texto:    C  a  p  i  t  u     e  r  a
Índice:   0  1  2  3  4  5  6  7  8  9

Entrada X:  [C, a, p, i, t, u, ' ', e]   (posições 0..7)
Alvo    Y:  [a, p, i, t, u, ' ', e, r]   (posições 1..8)  ← deslocado 1

Previsão em cada posição:
  pos 0: dado [C]          → prever 'a'
  pos 1: dado [C, a]       → prever 'p'
  pos 2: dado [C, a, p]    → prever 'i'
  ...
```

### Janela Deslizante

Percorremos o corpus com um stride de 1, gerando um sample por posição:

```
corpus: [t0, t1, t2, t3, t4, t5, t6, t7, ...]
         ├──────────── ctx ──────────┤
sample0: X=[t0..tL-1]   Y=[t1..tL]
          ├──────────── ctx ──────────┤
sample1: X=[t1..tL]     Y=[t2..tL+1]
```

Isso maximiza o uso do corpus sem repetições.


In [ ]:
class TextDataset(Dataset):
    """
    Dataset de linguagem com janela deslizante.

    Cada item é um par (X, Y) onde:
      - X[i] = sequência de `context_len` tokens a partir do índice i
      - Y[i] = X[i] deslocado 1 posição (o 'próximo' de cada posição)

    Args:
        data        : tensor 1D com todos os tokens do split
        context_len : tamanho da janela de contexto
    """

    def __init__(self, data: torch.Tensor, context_len: int):
        self.data        = data
        self.context_len = context_len

    def __len__(self):
        # Total de amostras possíveis
        return len(self.data) - self.context_len

    def __getitem__(self, idx: int):
        x = self.data[idx       : idx + self.context_len    ]
        y = self.data[idx + 1   : idx + self.context_len + 1]
        return x, y


# ── Testar o dataset ─────────────────────────────────────────────────────────
CONTEXT_LEN = 256  # valor provisório para teste (será redefinido abaixo)

ds_teste = TextDataset(train_data, CONTEXT_LEN)
x0, y0  = ds_teste[0]

print(f"Amostras no dataset : {len(ds_teste):,}")
print(f"Shape de X          : {x0.shape}")
print(f"Shape de Y          : {y0.shape}")
print()
print("X[:40] decodificado :", repr(tokenizer.decode(x0[:40].tolist())))
print("Y[:40] decodificado :", repr(tokenizer.decode(y0[:40].tolist())))
print()
print("► Y é X deslocado 1 posição para frente ✓")


In [ ]:
# ============================================================
#  HIPERPARÂMETROS DO MODELO
#  (otimizados para Google Colab T4 ~16 GB VRAM)
# ============================================================
CONTEXT_LEN    = 256   # janela de contexto (tokens)
D_MODEL        = 256   # dimensão dos embeddings
N_HEADS        = 8     # cabeças de atenção (D_MODEL // N_HEADS = 32)
N_LAYERS       = 6     # blocos Transformer empilhados
D_FF           = 1024  # dimensão interna do Feed-Forward (4 × D_MODEL)
DROPOUT        = 0.10  # taxa de dropout
BATCH_SIZE     = 128   # amostras por batch
LEARNING_RATE  = 3e-4  # taxa inicial (AdamW)
MAX_EPOCHS     = 40    # épocas de treinamento

# ── Flags de otimização ──────────────────────────────────────
USE_AMP     = device.type == "cuda"   # Mixed Precision só funciona em GPU
USE_COMPILE = int(torch.__version__.split(".")[0]) >= 2  # torch.compile ≥ 2.0
# ============================================================

# ── DataLoader otimizado ─────────────────────────────────────────────────────
#
#  num_workers=4        → 4 processos paralelos carregam batches em background
#                         enquanto a GPU treina o batch atual
#
#  persistent_workers   → mantém os workers vivos entre épocas, evitando o
#                         overhead de fork/join a cada época (~0.5s economizados)
#
#  prefetch_factor=2    → cada worker pré-carrega 2 batches na memória,
#                         garantindo que a GPU nunca espere dados
#
#  pin_memory=True      → aloca os tensores em memória fixada (page-locked),
#                         acelerando a transferência CPU→GPU via DMA
#
_is_cuda = device.type == "cuda"
_nw      = 4 if _is_cuda else 0
_pf      = 2 if _is_cuda else None

train_ds = TextDataset(train_data, CONTEXT_LEN)
val_ds   = TextDataset(val_data,   CONTEXT_LEN)

train_loader = DataLoader(
    train_ds,
    batch_size        = BATCH_SIZE,
    shuffle           = True,
    num_workers       = _nw,
    pin_memory        = _is_cuda,
    persistent_workers= _is_cuda,   # evita fork/join a cada época
    prefetch_factor   = _pf,        # pré-carrega 2× batches por worker
)
val_loader = DataLoader(
    val_ds,
    batch_size        = BATCH_SIZE,
    shuffle           = False,
    num_workers       = _nw,
    pin_memory        = _is_cuda,
    persistent_workers= _is_cuda,
    prefetch_factor   = _pf,
)

# Métricas rápidas
x_b, y_b = next(iter(train_loader))
print(f"Batches de treino      : {len(train_loader):,}")
print(f"Batches de validação   : {len(val_loader):,}")
print(f"Shape de um batch X    : {x_b.shape}  → (batch, context_len)")
print(f"Shape de um batch Y    : {y_b.shape}")
print(f"\nFlags de otimização:")
print(f"  USE_AMP     : {USE_AMP}   (Mixed Precision)")
print(f"  USE_COMPILE : {USE_COMPILE}   (torch.compile)")
print(f"  num_workers : {_nw}")


---
## 🏗️ Módulo 4 — Arquitetura: Blocos do Transformer

Vamos construir o modelo **de baixo para cima**, peça por peça.

### Diagrama de um Bloco Transformer (Pre-LN)

```
  Entrada x  ──────────────────────────────────┐
       │                                        │ (residual)
  LayerNorm                                     │
       │                                        │
  Causal Self-Attention                         │
       │                                        │
       └──────────── + ◄──────────────────────┘
                     │
                     ├──────────────────────────┐
                     │                          │ (residual)
                  LayerNorm                     │
                     │                          │
                FeedForward                     │
                     │                          │
                     └──── + ◄─────────────────┘
                            │
                         Saída x'
```

### Por que conexões residuais?

`saída = x + sublayer(x)` → o gradiente tem um *atalho* direto para fluir pelas somas, resolvendo o **problema do gradiente que desaparece** em redes muito profundas.

### Pre-LN vs Post-LN

- **Post-LN** (paper original): LayerNorm *depois* da soma residual → instável em redes profundas
- **Pre-LN** (GPT-2, LLaMA): LayerNorm *antes* da sublayer → treino mais estável ✅


In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention com máscara causal (unidirecional).

    Cada token pode atender APENAS aos tokens anteriores a ele.
    Isso permite geração autoregressiva: o modelo nunca "vê o futuro".

    Matemática de uma cabeça:
    ─────────────────────────
        Q = X·W_Q,  K = X·W_K,  V = X·W_V

        Attention(Q,K,V) = softmax( Q·Kᵀ / √d_k  +  mask ) · V

        onde mask[i,j] = 0 se j ≤ i, -∞ se j > i

    Com N_HEADS cabeças, rodamos isso em paralelo com subvetores de
    tamanho d_head = d_model // n_heads e concatenamos o resultado.
    """

    def __init__(self, d_model: int, n_heads: int,
                 context_len: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model deve ser divisível por n_heads"

        self.d_model     = d_model
        self.n_heads     = n_heads
        self.d_head      = d_model // n_heads   # dimensão por cabeça

        # Q, K, V em uma só projeção (3× mais eficiente)
        self.qkv_proj   = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj   = nn.Linear(d_model, d_model,     bias=False)

        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # Máscara causal: triângulo inferior de 1s
        # register_buffer → vai junto com o modelo no .to(device), mas não
        # é um parâmetro treinável
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_len, context_len))
              .view(1, 1, context_len, context_len)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape   # Batch | Seq_len | d_model

        # ── 1. Calcular Q, K, V de uma vez ───────────────────────────────
        qkv      = self.qkv_proj(x)                       # (B, T, 3*C)
        Q, K, V  = qkv.split(self.d_model, dim=-1)        # cada: (B, T, C)

        # ── 2. Reshape → múltiplas cabeças (B, n_heads, T, d_head) ───────
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        # ── 3. Attention scores: QKᵀ / √d_k ─────────────────────────────
        scale  = math.sqrt(self.d_head)
        scores = (Q @ K.transpose(-2, -1)) / scale        # (B, n_heads, T, T)

        # ── 4. Máscara causal: posições futuras → -inf ────────────────────
        mask   = self.causal_mask[:, :, :T, :T]
        scores = scores.masked_fill(
            mask == 0,
            torch.finfo(scores.dtype).min   # seguro em float16 e float32
        )

        # ── 5. Softmax + dropout ─────────────────────────────────────────
        attn_w = F.softmax(scores, dim=-1)
        attn_w = self.attn_drop(attn_w)

        # ── 6. Soma ponderada dos Values ─────────────────────────────────
        out = attn_w @ V                                   # (B, n_heads, T, d_head)

        # ── 7. Concatenar cabeças → projeção final ────────────────────────
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, d_model)
        out = self.resid_drop(self.out_proj(out))

        return out


print("✅ CausalSelfAttention definido")


### 🔍 Visualizando a Máscara Causal

A máscara impede que a posição `i` "veja" as posições `j > i`. Na prática, substituímos os logits proibidos por `-∞`, que após o **softmax** viram exatamente `0` de probabilidade.

```
Scores brutos (T×T):         Após máscara (-inf):       Após softmax:
┌─────────────────┐          ┌─────────────────┐        ┌─────────────────┐
│  s00  s01  s02  │          │  s00  -∞   -∞   │        │ 1.00  0.00  0.00│
│  s10  s11  s12  │  ─────►  │  s10  s11  -∞   │ ─────► │ p10   p11  0.00 │
│  s20  s21  s22  │          │  s20  s21  s22  │        │ p20   p21  p22  │
└─────────────────┘          └─────────────────┘        └─────────────────┘
```

Token 0 só atende a si mesmo. Token 2 atende a todos os anteriores.


In [ ]:
T_viz = 10   # tamanho da sequência para visualização

mask_bool = torch.tril(torch.ones(T_viz, T_viz))
scores_ex = torch.randn(T_viz, T_viz)
scores_m  = scores_ex.masked_fill(mask_bool == 0, float("-inf"))
attn_viz  = F.softmax(scores_m, dim=-1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── Máscara booleana ─────────────────────────────────────────────────────────
axes[0].imshow(mask_bool, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axes[0].set_title("1. Máscara Causal\n(1=permitido, 0=bloqueado)", fontsize=11)
for i in range(T_viz):
    for j in range(T_viz):
        axes[0].text(j, i, "✓" if mask_bool[i, j] else "✗",
                     ha="center", va="center", fontsize=9,
                     color="white" if mask_bool[i, j] else "#aaa")

# ── Scores mascarados ────────────────────────────────────────────────────────
sm = scores_m.clone()
sm[sm == float("-inf")] = float("nan")
im1 = axes[1].imshow(sm.numpy(), cmap="RdYlGn", aspect="auto")
axes[1].set_title("2. Scores mascarados\n(NaN onde era -∞)", fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# ── Pesos de atenção ─────────────────────────────────────────────────────────
im2 = axes[2].imshow(attn_viz.detach().numpy(), cmap="hot", vmin=0, aspect="auto")
axes[2].set_title("3. Pesos após Softmax\n(posições futuras = 0)", fontsize=11)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel("Posição K (chave)")
    ax.set_ylabel("Posição Q (query)")

plt.suptitle("Pipeline da Máscara Causal", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
class FeedForward(nn.Module):
    """
    Rede Feed-Forward posição-a-posição.

    Aplicada INDEPENDENTEMENTE a cada token (sem troca de informação entre
    posições). É aqui que o modelo "pensa" sobre cada representação depois
    da atenção ter coletado contexto.

    Estrutura:
        d_model  →  [Linear]  →  4·d_model  →  [GELU]  →  [Linear]  →  d_model

    Por que 4×?
        Proporção empírica do GPT original (Radford et al., 2018).
        A expansão cria capacidade para representações mais ricas.

    Por que GELU em vez de ReLU?
        GELU(x) ≈ x·Φ(x) (Φ = CDF da normal).
        É diferenciável em todo ponto → fluxo de gradiente mais suave.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


print("✅ FeedForward definido")


In [ ]:
class TransformerBlock(nn.Module):
    """
    Um bloco Transformer completo com Pre-LayerNorm.

    Fluxo:
        x  →  LN  →  CausalSelfAttention  →  +x   (residual 1)
           →  LN  →  FeedForward          →  +x   (residual 2)

    Pre-LN (normalização ANTES da sublayer) é mais estável para treinar
    do que o Post-LN do paper original, especialmente em redes profundas.
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 context_len: int, dropout: float):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, context_len, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))   # atenção + residual
        x = x + self.ff(self.ln2(x))     # feed-forward + residual
        return x


print("✅ TransformerBlock definido")


---
## 🧠 Módulo 5 — Modelo Completo: MiniGPT

Agora conectamos todas as peças:

```
Tokens (B, T)
     │
     ▼
Token Embedding (vocab_size → d_model)   ← "o que é este token?"
+
Positional Embedding (T → d_model)       ← "onde está este token?"
     │
     ▼
Dropout
     │
     ▼
Transformer Block × N_LAYERS             ← "o que cada token significa no contexto?"
     │
     ▼
LayerNorm final
     │
     ▼
Linear Head (d_model → vocab_size)       ← "qual é o próximo token?"
     │
     ▼
Logits (B, T, vocab_size)
```

### Weight Tying

Compartilhamos os pesos entre a camada `Token Embedding` e o `Linear Head`. Isso reduz parâmetros e, empiricamente, melhora a performance — faz sentido intuitivo: tokens semanticamente próximos devem ter embeddings parecidos *e* logits parecidos.

### Inicialização de Pesos (estilo GPT-2)

- Pesos das camadas lineares: $\mathcal{N}(0, 0.02)$
- Projeções de saída das sublayers: $\mathcal{N}(0, 0.02 / \sqrt{2 \cdot N_{layers}})$ (escala pela profundidade)


In [ ]:
class MiniGPT(nn.Module):
    """
    Transformer Decoder-Only para geração de texto (estilo GPT).

    Args:
        vocab_size  : tamanho do vocabulário (número de tokens únicos)
        context_len : comprimento máximo da janela de contexto
        d_model     : dimensão dos embeddings
        n_heads     : número de cabeças de atenção
        n_layers    : número de blocos Transformer
        d_ff        : dimensão interna do Feed-Forward
        dropout     : taxa de dropout
    """

    def __init__(
        self,
        vocab_size : int,
        context_len: int,
        d_model    : int,
        n_heads    : int,
        n_layers   : int,
        d_ff       : int,
        dropout    : float,
    ):
        super().__init__()
        self.context_len = context_len

        # ── Embeddings ────────────────────────────────────────────────────
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(context_len, d_model)
        self.drop      = nn.Dropout(dropout)

        # ── Pilha de blocos Transformer ───────────────────────────────────
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, context_len, dropout)
            for _ in range(n_layers)
        ])

        # ── Saída ─────────────────────────────────────────────────────────
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # ── Inicialização de pesos ────────────────────────────────────────
        self.apply(self._init_weights)

        # Escalar projeções de saída pela profundidade (GPT-2)
        for name, p in self.named_parameters():
            if name.endswith("out_proj.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * n_layers))

        # Weight tying: head compartilha pesos com token_emb
        self.head.weight = self.token_emb.weight

        total = self.count_params()
        print(f"✅ MiniGPT criado  —  {total:,} parâmetros treináveis")

    # ── Inicialização padrão ──────────────────────────────────────────────────
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, 0.0, 0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, 0.0, 0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # ── Forward ───────────────────────────────────────────────────────────────
    def forward(
        self,
        idx    : torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ):
        B, T = idx.shape
        assert T <= self.context_len, f"Sequência longa demais: {T} > {self.context_len}"

        # Posições: [0, 1, ..., T-1]
        pos = torch.arange(T, device=idx.device)

        # Soma de embeddings de token + posição
        x = self.drop(self.token_emb(idx) + self.pos_emb(pos))  # (B, T, d_model)

        # Blocos Transformer
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)          # (B, T, vocab_size)

        # Loss (somente se targets fornecidos)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

    # ── Geração autoregressiva ────────────────────────────────────────────────
    @torch.no_grad()
    def generate(
        self,
        prompt        : str,
        tokenizer     : CharTokenizer,
        max_new_tokens: int   = 200,
        temperature   : float = 1.0,
        top_k         : int   = 50,
    ) -> str:
        """
        Gera texto autoregressivamente a partir de um prompt.

        Args:
            temperature : controla a aleatoriedade
                            < 1 → mais conservador/repetitivo
                            = 1 → distribuição do modelo
                            > 1 → mais criativo/aleatório
            top_k       : considera só os K tokens mais prováveis
                          (evita samplear tokens muito improváveis)
        """
        self.eval()
        idx = torch.tensor(
            tokenizer.encode(prompt), dtype=torch.long, device=device
        ).unsqueeze(0)   # (1, T)

        for _ in range(max_new_tokens):
            # Trunca janela se necessário
            idx_ctx = idx[:, -self.context_len:]

            logits, _ = self(idx_ctx)
            logits = logits[:, -1, :]       # último token: (1, vocab_size)
            logits = logits / temperature   # escalar por temperatura

            # Top-k: zera logits fora dos k maiores
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

            probs      = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (1,1)
            idx        = torch.cat([idx, next_token], dim=1)

        return tokenizer.decode(idx[0].tolist())


print("✅ MiniGPT definido")


In [ ]:
# ── Instanciar o modelo ──────────────────────────────────────────────────────
model = MiniGPT(
    vocab_size  = tokenizer.vocab_size,
    context_len = CONTEXT_LEN,
    d_model     = D_MODEL,
    n_heads     = N_HEADS,
    n_layers    = N_LAYERS,
    d_ff        = D_FF,
    dropout     = DROPOUT,
).to(device)

# ── Distribuição de parâmetros ────────────────────────────────────────────────
print("\n─" * 30)
print("Distribuição de parâmetros:")
print("─" * 30)
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name:12s} : {params:>10,}")
print("─" * 30)
total = model.count_params()
print(f"  {'TOTAL':12s} : {total:>10,}")

# ── Smoke test do forward ─────────────────────────────────────────────────────
print("\nSmoke test:")
x_t = torch.randint(0, tokenizer.vocab_size, (2, CONTEXT_LEN)).to(device)
y_t = torch.randint(0, tokenizer.vocab_size, (2, CONTEXT_LEN)).to(device)
logits, loss = model(x_t, y_t)
print(f"  Input shape  : {x_t.shape}")
print(f"  Output shape : {logits.shape}")
print(f"  Loss inicial : {loss.item():.4f}  (esperado ≈ {math.log(tokenizer.vocab_size):.4f})")


In [ ]:
# ── torch.compile — kernels fusionados (PyTorch 2.x, GPU compatível) ─────────
#
# torch.compile(inductor) gera kernels CUDA em tempo de execução.
# Em algumas GPUs do Kaggle isso falha com cudaErrorNoKernelImageForDevice,
# corrompendo o contexto CUDA. Por isso:
#   1. No Kaggle: TORCHDYNAMO_DISABLE=1 já foi definido na célula de ambiente.
#   2. Em qualquer ambiente: testamos ANTES de substituir `model`.
#   3. Só atribuímos model = compiled se o forward de validação passou.
#
import os

_compile_possivel = (
    USE_COMPILE
    and device.type == "cuda"
    and os.environ.get("TORCHDYNAMO_DISABLE") != "1"
)

if _compile_possivel:
    try:
        _model_compilado = torch.compile(model, mode="default")
        # Validação: forward curto antes de substituir model
        _x_val = torch.zeros(1, 8, dtype=torch.long, device=device)
        with torch.no_grad():
            _model_compilado(_x_val)
        # Só chega aqui se funcionou
        model = _model_compilado
        del _model_compilado
        print("✅ torch.compile ativado — 1ª época será mais lenta (compilando kernels)")
    except Exception as e:
        USE_COMPILE = False
        # NÃO substituímos model — ele continua limpo
        print(f"⚠️  torch.compile desabilitado: {type(e).__name__}")
        print("   Continuando sem compilação — AMP ainda oferece ~2× de ganho.")
else:
    USE_COMPILE = False
    razao = "CPU" if device.type != "cuda" else (
            "PyTorch < 2.0" if not USE_COMPILE else
            "desabilitado no Kaggle")
    print(f"ℹ️  torch.compile não ativado ({razao})")


---
## 💾 Módulo 5c — Persistência do Modelo

Treinar um modelo do zero leva tempo. Para que o resultado seja aproveitável em
**outras aulas, outros notebooks e outros computadores**, precisamos salvar não
apenas os pesos, mas tudo que for necessário para reconstruir o modelo sem
depender do corpus ou do ambiente original.

### O problema do `state_dict` simples

O PyTorch salva os pesos com `torch.save(model.state_dict(), "modelo.pt")`, mas
este arquivo sozinho é inútil sem saber:

```
state_dict  (pesos)          →  "quais são os números"
+ config    (hiperparâmetros) →  "como montar a arquitetura"
+ vocab     (tokenizador)     →  "como converter texto ↔ inteiros"
+ metadata  (histórico)       →  "qual foi a melhor época, a loss final..."
```

### Solução: Bundle Completo

Salvamos tudo em um único arquivo `.pt` que é **autossuficiente**:

```
minigpt_domcasmurro_bundle.pt
│
├── model_state   → state_dict() com todos os pesos
├── config        → vocab_size, context_len, d_model, n_heads, n_layers, d_ff
├── vocab         → chars, char_to_idx, idx_to_char
├── training      → melhor época, val_loss, histórico completo
└── metadata      → timestamp, versão, corpus usado
```

### Google Drive como storage persistente

No Colab, tudo em `/content/` é **apagado** ao fechar a sessão. O Drive é a
única forma de persistir arquivos entre sessões sem fazer download manual.

```
Sessão A (treino)       Sessão B (aula seguinte)
─────────────────       ─────────────────────────
train() → bundle  ──►  Drive  ──►  load_bundle()
                                   model.generate(...)
```

### Checkpoint de Retomada

Se o Colab desconectar no meio do treino, podemos retomar do último checkpoint
salvo, sem perder o progresso:

```
época 1 → save checkpoint
época 2 → save checkpoint
época 3 → DESCONECTOU
época 3 → retomar de época 2 ✓   (sem recomeçar do zero)
```


In [ ]:
import json
from datetime import datetime


# ── Nomes de arquivo e caminhos (usam OUTPUT_DIR detectado acima) ─────────────
BUNDLE_NAME  = "minigpt_domcasmurro_bundle.pt"
CKPT_NAME    = "minigpt_checkpoint.pt"
BUNDLE_PATH  = os.path.join(OUTPUT_DIR, BUNDLE_NAME)   # caminho final do bundle
CKPT_PATH    = os.path.join(OUTPUT_DIR, CKPT_NAME)     # caminho final do checkpoint


def save_bundle(
    model,
    tokenizer,
    config      : dict,
    history     : dict,
    best_epoch  : int,
    best_val    : float,
    path        : str = None,
) -> str:
    """
    Salva bundle autossuficiente: pesos + config + vocab + histórico.
    Funciona em Kaggle, Colab e local — usa OUTPUT_DIR automaticamente.
    """
    if path is None:
        path = BUNDLE_PATH

    raw = model._orig_mod if hasattr(model, "_orig_mod") else model

    bundle = {
        "model_state": raw.state_dict(),
        "config"     : config,
        "vocab": {
            "chars"      : tokenizer.chars,
            "char_to_idx": tokenizer.char_to_idx,
            "idx_to_char": {str(k): v for k, v in tokenizer.idx_to_char.items()},
        },
        "training": {
            "history"   : history,
            "best_epoch": best_epoch,
            "best_val"  : best_val,
            "n_params"  : raw.count_params(),
        },
        "metadata": {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "corpus"   : "Dom Casmurro — Machado de Assis",
            "version"  : "1.0",
            "ambiente" : "kaggle" if IS_KAGGLE else ("colab" if IS_COLAB else "local"),
        },
    }

    torch.save(bundle, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  💾 Bundle salvo : {path}  ({size_mb:.1f} MB)")

    if IS_KAGGLE:
        print(f"  📥 Para baixar  : aba 'Output' → '{BUNDLE_NAME}' → Download")

    return path


def load_bundle(path: str = None):
    """
    Carrega bundle e reconstrói modelo + tokenizador prontos para uso.
    Retorna (model, tokenizer, info_dict).
    """
    if path is None:
        path = BUNDLE_PATH

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Bundle não encontrado: {path}\n"
            f"Execute o treino primeiro ou verifique o caminho."
        )

    bundle = torch.load(path, map_location=device, weights_only=False)

    class _BundleTokenizer:
        def __init__(self, v):
            self.chars       = v["chars"]
            self.vocab_size  = len(self.chars)
            self.char_to_idx = v["char_to_idx"]
            self.idx_to_char = {int(k): ch for k, ch in v["idx_to_char"].items()}
        def encode(self, text):
            return [self.char_to_idx[c] for c in text if c in self.char_to_idx]
        def decode(self, ids):
            return "".join(self.idx_to_char[i] for i in ids)
        def __len__(self):
            return self.vocab_size

    tok  = _BundleTokenizer(bundle["vocab"])
    cfg  = bundle["config"]
    m    = MiniGPT(**cfg, dropout=0.0).to(device)
    m.load_state_dict(bundle["model_state"])
    m.eval()

    tr   = bundle["training"]
    meta = bundle["metadata"]
    info = {**tr, **meta, "config": cfg}

    print(f"✅ Bundle carregado : {path}")
    print(f"   Corpus           : {meta['corpus']}")
    print(f"   Treinado em      : {meta['timestamp']}")
    print(f"   Ambiente origem  : {meta.get('ambiente','—')}")
    print(f"   Parâmetros       : {tr['n_params']:,}")
    print(f"   Melhor val_loss  : {tr['best_val']:.4f}  (época {tr['best_epoch']})")

    return m, tok, info


def save_resume_checkpoint(model, optimizer, scheduler, scaler,
                           epoch, history, best_val):
    """Salva estado completo para retomar treino interrompido."""
    raw  = model._orig_mod if hasattr(model, "_orig_mod") else model
    ckpt = {
        "epoch"       : epoch,
        "model_state" : raw.state_dict(),
        "optim_state" : optimizer.state_dict(),
        "sched_state" : scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "history"     : history,
        "best_val"    : best_val,
    }
    torch.save(ckpt, CKPT_PATH)


def load_resume_checkpoint(model, optimizer, scheduler, scaler):
    """
    Tenta carregar checkpoint de retomada.
    Retorna (start_epoch, history, best_val).
    """
    path = CKPT_PATH if os.path.exists(CKPT_PATH) else None

    if path is None:
        print("ℹ️  Nenhum checkpoint — iniciando do zero.")
        return 1, {"train_loss": [], "val_loss": [], "lr": [], "epoch_time": []}, float("inf")

    ckpt = torch.load(path, map_location=device, weights_only=False)
    raw  = model._orig_mod if hasattr(model, "_orig_mod") else model
    raw.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    scaler.load_state_dict(ckpt["scaler_state"])

    start = ckpt["epoch"] + 1
    print(f"✅ Retomando da época {start}  (val_loss anterior: {ckpt['best_val']:.4f})")
    return start, ckpt["history"], ckpt["best_val"]


print("✅ Funções de persistência (ambiente:", "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local"), ")")
print(f"   BUNDLE_PATH = {BUNDLE_PATH}")
print(f"   CKPT_PATH   = {CKPT_PATH}")


---
## ⚡ Módulo 6a — Otimizações de Performance

Antes do loop de treino, aplicamos três técnicas que **reduzem o tempo por época
de ~25 min para ~5–8 min no T4**, sem alterar a matemática ou os resultados do modelo.

---

### Otimização 1 — Mixed Precision (AMP)

Por padrão, o PyTorch usa `float32` (32 bits) para todos os tensores. GPUs modernas
possuem **Tensor Cores** que operam nativamente em `float16` (16 bits), entregando
**2–4× mais operações por segundo** com metade do uso de memória.

O módulo `torch.amp` gerencia a precisão automaticamente:

```
forward pass  → float16  (rápido, menos VRAM)
backward pass → float32  (preciso, evita underflow do gradiente)
```

O `GradScaler` resolve o principal risco do float16 — **underflow de gradiente** —
multiplicando a loss por um fator de escala antes do backward e depois desfazendo
a escala nos pesos, mantendo a estabilidade numérica:

```
Loss × escala  →  backward  →  gradientes / escala  →  optimizer.step()
     ↑                                                        ↑
 float16 seguro                                         float32 correto
```

---

### Otimização 2 — `torch.compile`

Introduzido no PyTorch 2.0, `torch.compile` compila o modelo em **kernels CUDA
fusionados** usando o compilador TorchInductor. Operações que antes eram executadas
separadamente (LayerNorm → Linear → GELU → Dropout) são combinadas em uma única
passagem pela memória da GPU:

```
Sem compile:    LayerNorm  →  Linear  →  GELU  →  Dropout
                 (4 leituras/escritas na VRAM)

Com compile:   [LayerNorm + Linear + GELU + Dropout]
                 (1 leitura/escrita fusionada)
```

> **Atenção:** a primeira época demora mais (compilando os kernels).
> Das seguintes em diante o ganho é de ~20–30% por época.

---

### Otimização 3 — DataLoader Paralelo

O gargalo mais subestimado em treinos curtos é o **pipeline de dados**. Sem
paralelismo, a GPU fica ociosa esperando o CPU preparar o próximo batch:

```
Sem otimização:   [CPU prepara batch] → [GPU treina] → [CPU prepara] → [GPU treina]
                   ████████████████       ████████       ████████        ████████

Com otimização:   [CPU prepara batch 2] durante [GPU treina batch 1]
                   ░░░░░░░░░░░░░░░░░░       ████████████████████
                         (paralelo!)
```

| Parâmetro | Valor | Efeito |
|---|---|---|
| `num_workers=4` | 4 processos paralelos | CPU e GPU trabalham simultaneamente |
| `persistent_workers=True` | workers permanecem vivos | elimina fork/join entre épocas |
| `prefetch_factor=2` | 2 batches pré-carregados | GPU nunca espera dados |
| `pin_memory=True` | memória fixada | transferência CPU→GPU via DMA direto |

---

### Resumo do ganho esperado no Colab T4

| Configuração | Tempo/época | Tempo total (60 épocas) |
|---|---|---|
| Original (sem otimizações) | ~25 min | ~25 h |
| + DataLoader paralelo | ~18 min | ~18 h |
| + AMP (float16) | ~9 min | ~9 h |
| + torch.compile | ~6–7 min | ~6–7 h |
| **Todas juntas** | **~5–7 min** | **~5–7 h** |


---
## 🏋️ Módulo 6 — Treinamento

### Função de Loss: Cross-Entropy

Para cada posição $t$, o modelo gera uma distribuição de probabilidade sobre o vocabulário. Queremos que a probabilidade do token correto $x_{t+1}$ seja alta:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(x_{t+1} \mid x_1, \ldots, x_t)$$

Uma perda aleatória seria $\ln(\text{vocab\_size}) \approx \ln(87) \approx 4.46$. Com um bom treino, conseguimos cair para ~1.2–1.5.

### Otimizador: AdamW

Adam com weight decay (L2 regularização). Não aplicamos weight decay em biases e LayerNorm — apenas nas matrizes de peso.

### Learning Rate Schedule: Cosine com Warmup

```
LR  ↑        ┌─╮
    │       ╱    ╲___________
    │      ╱
    │   ╱
    └──────────────────────── steps
     warmup      cosine decay
```

O **warmup** evita instabilidade no início quando os pesos são aleatórios.
O **cosine decay** reduz suavemente o LR no final para convergência fina.

### Gradient Clipping

Limitamos a norma máxima dos gradientes a 1.0 para evitar explosão de gradiente — comum em Transformers sem este controle.


In [ ]:
def build_optimizer(model, lr: float, wd: float = 0.1):
    """
    AdamW com weight decay seletivo.
    - Matrizes (Linear, Embedding): weight decay ativado
    - Biases e LayerNorm: sem weight decay
    """
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim < 2 or "bias" in name or "ln" in name or "norm" in name:
            no_decay.append(p)
        else:
            decay.append(p)

    groups = [
        {"params": decay,    "weight_decay": wd},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95), eps=1e-8)


def build_scheduler(optimizer, warmup_steps: int, total_steps: int):
    """Cosine schedule com warmup linear."""
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate_loss(model, loader, max_batches: int = 50) -> float:
    """
    Calcula a loss média de validação.
    Usa autocast para consistência com o treino e menor uso de VRAM.
    """
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        # autocast na validação: mesma precisão do treino, sem GradScaler
        with autocast(device_type=device.type, enabled=USE_AMP):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


print("✅ Funções de treinamento definidas")


In [ ]:
def treinar(
    model,
    train_loader,
    val_loader,
    epochs      : int,
    lr          : float,
    sample_freq : int  = 10,    # gerar sample a cada N épocas
    ckpt_freq   : int  = 5,     # salvar checkpoint de retomada a cada N épocas
    resume      : bool = True,  # tentar retomar de checkpoint anterior
) -> dict:
    """
    Loop principal de treinamento com:
      - Mixed Precision (AMP) via GradScaler + autocast
      - Gradient clipping
      - Cosine LR schedule com warmup
      - Salvamento automático de bundle completo (pesos + vocab + config)
      - Checkpoint periódico para retomada em caso de desconexão
      - Suporte a resume de treino interrompido
    """
    # Config do modelo para o bundle
    _config = dict(
        vocab_size  = tokenizer.vocab_size,
        context_len = CONTEXT_LEN,
        d_model     = D_MODEL,
        n_heads     = N_HEADS,
        n_layers    = N_LAYERS,
        d_ff        = D_FF,
    )

    optimizer    = build_optimizer(model, lr)
    total_steps  = epochs * len(train_loader)
    warmup_steps = total_steps // 20
    scheduler    = build_scheduler(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler(device=device.type, enabled=USE_AMP)

    history       = {"train_loss": [], "val_loss": [], "lr": [], "epoch_time": []}
    best_val_loss = float("inf")
    best_epoch    = 0
    t0            = time.time()

    # ── Tentar retomar treino anterior ────────────────────────────────────
    start_epoch = 1
    if resume:
        start_epoch, history, best_val_loss = load_resume_checkpoint(
            model, optimizer, scheduler, scaler
        )
        best_epoch = start_epoch - 1

    # ── Ajustar total_steps para epochs restantes ─────────────────────────
    remaining  = epochs - (start_epoch - 1)
    if remaining <= 0:
        print(f"✅ Treino já completo ({epochs} épocas). Carregue o bundle.")
        return history

    loss_inicial = evaluate_loss(model, val_loader, max_batches=20)
    amp_str      = "AMP float16" if USE_AMP else "float32"
    cmp_str      = "torch.compile ON" if USE_COMPILE else "torch.compile OFF"
    print(f"Loss atual           : {loss_inicial:.4f}")
    print(f"Loss teórica (random): {math.log(tokenizer.vocab_size):.4f}")
    print(f"Precisão             : {amp_str}  |  {cmp_str}")
    drive_str = DRIVE_DIR if USAR_DRIVE else "local apenas"
    print(f"Salvamento           : {drive_str}")
    print(f"\nIniciando época {start_epoch} → {epochs}  "
          f"({remaining} épocas restantes  ·  "
          f"{remaining * len(train_loader):,} steps)\n")
    print(f"{'Época':>6}  {'Train':>7}  {'Val':>7}  {'LR':>9}  {'t/época':>8}")
    print("─" * 50)

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        batch_losses = []
        t_epoch = time.time()

        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=device.type, enabled=USE_AMP):
                _, loss = model(x, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            batch_losses.append(loss.item())

        # ── Métricas ──────────────────────────────────────────────────────
        train_loss = float(np.mean(batch_losses))
        val_loss   = evaluate_loss(model, val_loader)
        cur_lr     = scheduler.get_last_lr()[0]
        epoch_secs = time.time() - t_epoch

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(cur_lr)
        history["epoch_time"].append(epoch_secs)

        # ── Melhor modelo → salvar bundle completo ────────────────────────
        saved = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch
            save_bundle(
                model, tokenizer, _config, history,
                best_epoch, best_val_loss,
                path=BUNDLE_PATH,
            )
            saved = " ✅"

        # ── Checkpoint periódico de retomada ──────────────────────────────
        if epoch % ckpt_freq == 0:
            save_resume_checkpoint(
                model, optimizer, scheduler, scaler,
                epoch, history, best_val_loss
            )

        print(
            f"{epoch:6d}  {train_loss:7.4f}  {val_loss:7.4f}"
            f"  {cur_lr:9.2e}  {epoch_secs:7.1f}s{saved}"
        )

        # ── Sample periódico ──────────────────────────────────────────────
        if epoch % sample_freq == 0:
            raw = model._orig_mod if hasattr(model, "_orig_mod") else model
            sample = raw.generate(
                "Capitu olhou para mim", tokenizer,
                max_new_tokens=120, temperature=0.8, top_k=40
            )
            print(f"\n  📝 [{epoch}] {sample[:180]}\n")

    # ── Resumo final ──────────────────────────────────────────────────────
    total_time  = time.time() - t0
    media_epoca = float(np.mean(history["epoch_time"]))
    print(f"\n✅ Treino concluído em {total_time/60:.1f} min")
    print(f"   Tempo médio/época : {media_epoca:.1f}s")
    print(f"   Melhor val_loss   : {best_val_loss:.4f}  (época {best_epoch})")
    print(f"   Bundle salvo em   : {BUNDLE_PATH}")

    return history


print("✅ Função `treinar` definida (AMP + persistência + resume)")


In [ ]:
# ── Executar o treinamento ────────────────────────────────────────────────────
#
# ⏱️  Tempo estimado por época (com todas as otimizações):
#   Kaggle T4   : ~6 min/época  →  60 épocas ≈ 6h  (cabe na sessão de 9h)
#   Colab T4    : ~6 min/época  →  60 épocas ≈ 6h  (pode exigir 2 sessões)
#   Kaggle P100 : ~14 min/época →  60 épocas ≈ 14h (usar resume em 2 sessões)
#   CPU (local) : ~90 min/época →  use MAX_EPOCHS = 5 para teste
#
# ⚠️  A PRIMEIRA ÉPOCA é mais lenta (torch.compile compilando kernels CUDA).
#
# 📥 No Kaggle: ao finalizar, baixe o bundle pela aba "Output" no painel direito.
# ─────────────────────────────────────────────────────────────────────────────
history = treinar(
    model,
    train_loader,
    val_loader,
    epochs      = MAX_EPOCHS,
    lr          = LEARNING_RATE,
    sample_freq = 10,
    ckpt_freq   = 5,
    resume      = True,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep = range(1, len(history["train_loss"]) + 1)
random_loss = math.log(tokenizer.vocab_size)

# ── Curvas de loss ────────────────────────────────────────────────────────────
axes[0].plot(ep, history["train_loss"], label="Train Loss",
             color="royalblue", linewidth=2)
axes[0].plot(ep, history["val_loss"],   label="Val Loss",
             color="tomato",    linewidth=2)
axes[0].axhline(y=random_loss, color="gray", linestyle="--",
                label=f"Loss aleatória ({random_loss:.2f})", linewidth=1.5)
axes[0].set_title("Curvas de Loss", fontsize=13)
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Anotação da melhora total
melhora = 100 * (1 - min(history["val_loss"]) / history["val_loss"][0])
axes[0].annotate(
    f"Melhora: {melhora:.1f}%",
    xy=(len(ep), min(history["val_loss"])),
    xytext=(len(ep) * 0.6, history["val_loss"][0] * 0.9),
    arrowprops=dict(arrowstyle="->", color="tomato"),
    color="tomato", fontsize=10
)

# ── Learning Rate schedule ────────────────────────────────────────────────────
axes[1].plot(ep, history["lr"], color="mediumseagreen", linewidth=2)
axes[1].set_title("Learning Rate Schedule", fontsize=13)
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Learning Rate")
axes[1].grid(alpha=0.3)

plt.suptitle("MiniGPT — Métricas de Treinamento", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("training_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Val loss final : {history['val_loss'][-1]:.4f}")
print(f"Val loss mínima: {min(history['val_loss']):.4f}")


---
## 📝 Módulo 7 — Geração de Texto

### Pipeline de Geração (autoregressiva)

```
prompt "Capitu era..."
        │
        ▼
encode → [10, 3, 55, ...]
        │
        ▼
 ┌──────────────────────────────────────┐
 │  loop (max_new_tokens vezes):        │
 │   1. forward(contexto)               │
 │   2. logits do ÚLTIMO token          │
 │   3. dividir por temperatura τ       │
 │   4. top-k filtering                 │
 │   5. softmax → probs                 │
 │   6. multinomial sample → next_token │
 │   7. append ao contexto              │
 └──────────────────────────────────────┘
        │
        ▼
decode → "Capitu era formosa, e eu..."
```

### Temperatura τ

| τ | Efeito |
|---|--------|
| 0.3 | Muito conservador, repetitivo |
| 0.7–0.9 | **Sweet spot** — coerente e variado |
| 1.2 | Mais criativo, pode "delirar" |
| 2.0 | Altamente aleatório |

### Top-k

Antes do softmax, zeramos os logits fora dos **k** maiores. Isso impede que o modelo escolha tokens muito improváveis (erros ortográficos, caracteres estranhos).


In [ ]:
# ── Carregar o melhor modelo salvo ───────────────────────────────────────────
model_inf, tokenizer_inf, info_bundle = load_bundle()

print(f"\nModelo pronto para geração:")
print(f"  val_loss  : {info_bundle['best_val']:.4f}")
print(f"  época     : {info_bundle['best_epoch']}")
print(f"  ambiente  : {info_bundle.get('ambiente','—')}")

# ── Experimento de temperatura ────────────────────────────────────────────────
prompts      = [
    "Capitu olhou para mim",
    "Era uma tarde de outubro e",
    "O tempo havia passado, mas",
    "Bentinho pensou que jamais",
]
temperatures = [0.5, 0.8, 1.2]

print("\n" + "=" * 72)
print("  GERAÇÃO DE TEXTO — MiniGPT | Dom Casmurro | Machado de Assis")
print("=" * 72)

for prompt in prompts:
    print(f"\n📌 Prompt: '{prompt}'")
    print("─" * 60)
    for tau in temperatures:
        gen = model_inf.generate(
            prompt, tokenizer_inf,
            max_new_tokens=160,
            temperature=tau,
            top_k=40,
        )
        continuation = gen[len(prompt):]
        print(f"  τ={tau:.1f} │ ...{continuation[:120]}")


---
## 🔬 Módulo 8 — Visualizando os Pesos de Atenção

Podemos inspecionar **o que o modelo presta atenção** ao processar um texto. Vamos extrair os pesos de atenção de cada cabeça nos blocos do Transformer.

Cada cabeça tende a especializar-se em diferentes padrões:
- Cabeças que acompanham a posição anterior (dependências locais)
- Cabeças que conectam sujeitos a verbos distantes
- Cabeças que rastreiam pontuação e início de frase

> **Como funciona?** Usamos **hooks** do PyTorch para capturar os tensores intermediários durante o `forward()` sem modificar o código do modelo.


In [ ]:
# ── Capturar pesos de atenção via hooks ──────────────────────────────────────
captured_attn = {}

def make_attn_hook(block_idx: int):
    """Cria um hook que recalcula e salva os pesos de atenção."""
    def hook(module, inp, out):
        with torch.no_grad():
            x    = inp[0]
            B, T, C = x.shape
            qkv  = module.qkv_proj(x)
            Q, K, _ = qkv.split(module.d_model, dim=-1)
            nh, dh = module.n_heads, module.d_head
            Q = Q.view(B, T, nh, dh).transpose(1, 2)
            K = K.view(B, T, nh, dh).transpose(1, 2)
            sc = (Q @ K.transpose(-2, -1)) / math.sqrt(dh)
            mask = module.causal_mask[:, :, :T, :T]
            sc   = sc.masked_fill(mask == 0, float("-inf"))
            w    = F.softmax(sc, dim=-1)
            captured_attn[f"bloco_{block_idx}"] = w.cpu()
    return hook

# Registrar hooks em todos os blocos
hooks = [
    blk.attn.register_forward_hook(make_attn_hook(i))
    for i, blk in enumerate(model.blocks)
]

# Forward com um trecho do texto
trecho_viz  = "Capitu era formosa, de olhos de ressaca"
tokens_viz  = tokenizer.encode(trecho_viz)
x_viz       = torch.tensor(tokens_viz, dtype=torch.long).unsqueeze(0).to(device)
with torch.no_grad():
    model(x_viz)

for h in hooks:
    h.remove()

# ── Plotar cabeças do bloco 0 ─────────────────────────────────────────────────
bloco    = "bloco_0"
attn_w   = captured_attn[bloco][0]   # (n_heads, T, T)
chars_viz = list(trecho_viz)
T_viz    = len(chars_viz)

ncols = 4
nrows = math.ceil(N_HEADS / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows))
axes = axes.flatten()

for hi in range(N_HEADS):
    ax = axes[hi]
    im = ax.imshow(attn_w[hi].numpy(), cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax.set_title(f"Cabeça {hi + 1}", fontsize=10)
    ax.set_xticks(range(T_viz))
    ax.set_yticks(range(T_viz))
    ax.set_xticklabels(chars_viz, fontsize=7, rotation=90)
    ax.set_yticklabels(chars_viz, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Ocultar eixos extras
for hi in range(N_HEADS, len(axes)):
    axes[hi].set_visible(False)

plt.suptitle(
    f'Pesos de Atenção — Bloco 1\n"{trecho_viz}"',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("attention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Observe como diferentes cabeças focam em padrões distintos!")


---
## Módulo 10 — Exemplo Pratico: Usando o Modelo Treinado

Depois do treino, o arquivo `best_minigpt.pt` contém os pesos do melhor modelo
(menor val_loss). Esta secao mostra como carregar e usar o modelo em qualquer
contexto, dentro ou fora do Colab.

---

### 10.1 Carregando o Modelo Salvo

O procedimento padrao do PyTorch para restaurar um modelo e:

1. Recriar a arquitetura com os **mesmos hiperparametros** usados no treino
2. Chamar `load_state_dict` com os pesos salvos
3. Mover o modelo para o dispositivo correto
4. Colocar em modo `.eval()` para desativar dropout

```python
import torch
from pathlib import Path

# Recriar tokenizador (precisa do mesmo corpus)
tokenizer = CharTokenizer(corpus)

# Recriar o modelo com os mesmos hiperparametros
model = MiniGPT(
    vocab_size  = tokenizer.vocab_size,
    context_len = CONTEXT_LEN,     # 256
    d_model     = D_MODEL,          # 256
    n_heads     = N_HEADS,          # 8
    n_layers    = N_LAYERS,         # 6
    d_ff        = D_FF,             # 1024
    dropout     = 0.0,              # zero no inference
).to(device)

# Carregar pesos
checkpoint = torch.load("best_minigpt.pt", map_location=device)
model.load_state_dict(checkpoint)
model.eval()

print("Modelo carregado com sucesso!")
print(f"Parametros: {model.count_params():,}")
```

> **Atencao:** o modelo e o tokenizador formam um par inseparavel. O vocabulario
> e construido a partir do corpus de treino, portanto o tokenizador deve ser
> recriado com o mesmo texto para que os indices batam.

---

### 10.2 Gerando Texto: Uso Basico

O metodo `model.generate()` aceita um **prompt** em texto puro e retorna a
continuacao gerada. Os parametros principais sao:

| Parametro | Tipo | Descricao | Valor padrao |
|---|---|---|---|
| `prompt` | str | Texto inicial (seed) | — |
| `max_new_tokens` | int | Quantos chars gerar alem do prompt | 200 |
| `temperature` | float | Criatividade (0.1 a 2.0) | 1.0 |
| `top_k` | int | Filtrar top-K tokens mais provaveis | 50 |

```python
# Exemplo mais simples possivel
texto = model.generate(
    prompt         = "Capitu olhou para mim",
    tokenizer      = tokenizer,
    max_new_tokens = 300,
    temperature    = 0.8,
    top_k          = 40,
)
print(texto)
```

**Saida tipica apos treino completo (60 epocas):**

```
Capitu olhou para mim com aqueles olhos de ressaca que eu nao sabia
definir. Era uma mistura de tristeza e alegria, como se o mar trouxesse
e levasse ao mesmo tempo. Dava-me vontade de perguntar o que havia, mas
as palavras morriam antes de chegar aos labios. Ela sorriu, e eu perdi
o fio do pensamento — como sempre acontecia diante de Capitu.
```

---

### 10.3 Experimento: Comparando Temperaturas

A **temperatura** e o parametro mais intuitivo para controlar o estilo
da geracao. Valores baixos tornam o modelo mais "seguro" e repetitivo;
valores altos, mais ousado e imprevisivel.

```python
prompt    = "Era uma vez um homem chamado"
temps     = [0.3, 0.7, 1.0, 1.4]

for tau in temps:
    out = model.generate(
        prompt,
        tokenizer,
        max_new_tokens=120,
        temperature=tau,
        top_k=40
    )
    continuation = out[len(prompt):]
    print(f"[tau={tau}] {continuation[:100]}")
    print()
```

**O que esperar:**

- `tau = 0.3` — Texto coerente mas com frases que se repetem.
  O modelo sempre escolhe o token mais provavel.
- `tau = 0.7` — Equilibrio entre coerencia e variedade. Recomendado
  para demonstracoes em sala de aula.
- `tau = 1.0` — Distribuicao original do modelo. Boa variedade,
  mas pode perder coerencia em sequencias longas.
- `tau = 1.4` — Criativo e imprevisivel. Pode gerar neologismos
  e construcoes inusuais, interessantes para analise linguistica.

---

### 10.4 Experimento: Comparando Prompts

O prompt funciona como **condicao inicial** da distribuicao de
probabilidade. Diferentes pontos de partida ativam diferentes
"memorias" do corpus:

```python
prompts = [
    # Abertura do capitulo 1 (o modelo "reconhece")
    "Capitu era",

    # Contexto emocional — testa geracao introspectiva
    "Nao ha tristeza maior do que",

    # Dialogo — testa estrutura conversacional
    '"Voce acredita em mim?" — perguntou ela',

    # Contexto de tempo — testa narrativa cronologica
    "Muitos anos depois, recordei que",

    # Fora do dominio — testa generalizacao
    "O computador processava os dados com",
]

for p in prompts:
    out = model.generate(p, tokenizer, max_new_tokens=150,
                         temperature=0.8, top_k=40)
    print(f"PROMPT : {p}")
    print(f"OUTPUT : {out[len(p):len(p)+120]}")
    print("-" * 60)
```

> **Exercicio para os alunos:** O ultimo prompt esta completamente fora
> do dominio do corpus. O que o modelo gera? O vocabulario gerado ainda
> parece machadiano? Por que isso acontece?

---

### 10.5 Salvando e Compartilhando o Modelo

Para reutilizar o modelo em outro notebook ou compartilhar com colegas,
basta salvar o checkpoint junto com os metadados necessarios para
recriacao:

```python
import json

# Pacote completo: pesos + config + vocabulario
bundle = {
    "model_state" : model.state_dict(),
    "config": {
        "vocab_size"  : tokenizer.vocab_size,
        "context_len" : CONTEXT_LEN,
        "d_model"     : D_MODEL,
        "n_heads"     : N_HEADS,
        "n_layers"    : N_LAYERS,
        "d_ff"        : D_FF,
    },
    "vocab": {
        "chars"       : tokenizer.chars,
        "char_to_idx" : tokenizer.char_to_idx,
        "idx_to_char" : {int(k): v
                         for k, v in tokenizer.idx_to_char.items()},
    },
}
torch.save(bundle, "minigpt_bundle.pt")
print("Bundle salvo!")
```

**Carregando em outro ambiente:**

```python
bundle = torch.load("minigpt_bundle.pt", map_location="cpu")
cfg    = bundle["config"]

# Recriar tokenizador a partir do vocabulario salvo
class BundleTokenizer:
    def __init__(self, vocab):
        self.chars       = vocab["chars"]
        self.vocab_size  = len(self.chars)
        self.char_to_idx = vocab["char_to_idx"]
        self.idx_to_char = {int(k): v
                            for k, v in vocab["idx_to_char"].items()}
    def encode(self, text):
        return [self.char_to_idx[c] for c in text
                if c in self.char_to_idx]
    def decode(self, ids):
        return "".join(self.idx_to_char[i] for i in ids)

tokenizer2 = BundleTokenizer(bundle["vocab"])
model2     = MiniGPT(**cfg, dropout=0.0)
model2.load_state_dict(bundle["model_state"])
model2.eval()

print(model2.generate("Capitu", tokenizer2,
                       max_new_tokens=100, temperature=0.8))
```

---

### 10.6 Perguntas para Reflexao

- Por que o modelo gera texto em portugues arcaico mesmo sem nunca
  ter recebido essa instrucao explicitamente?
- O que aconteceria se treinassemos com um corpus em ingles mas
  usassemos um prompt em portugues?
- Como a loss de validacao se correlaciona com a qualidade perceptual
  do texto gerado?
- Qual seria o impacto de aumentar `context_len` de 256 para 512?
  Pense em termos de memoria, velocidade e qualidade de geracao.
- Este modelo "entende" portugues ou apenas aprende padroes
  estatisticos de sequencias de caracteres?


---
## 📂 Módulo 11 — Carregando o Modelo em Outra Aula

Esta seção é **autocontida**: pode ser copiada para qualquer notebook para
usar o modelo treinado, sem precisar rodar todo o pipeline de treinamento.

Basta ter o arquivo `minigpt_domcasmurro_bundle.pt` salvo no Google Drive.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA STANDALONE — USE EM QUALQUER NOTEBOOK                           ║
# ║  Pré-requisito: bundle salvo no Drive após o treino                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, math, torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Redefinir arquitetura (necessário para carregar o state_dict) ─────────────
# Copie as classes CausalSelfAttention, FeedForward, TransformerBlock e MiniGPT
# para este notebook OU importe de um módulo compartilhado.
# Aqui assumimos que elas já estão definidas neste kernel.

# ── Montar Drive e carregar bundle ────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

BUNDLE_PATH = "/content/drive/MyDrive/MiniGPT_DomCasmurro/minigpt_domcasmurro_bundle.pt"

bundle = torch.load(BUNDLE_PATH, map_location=device, weights_only=False)

# ── Reconstruir tokenizador a partir do vocabulário salvo ─────────────────────
class BundleTokenizer:
    """Tokenizador reconstruído diretamente do bundle, sem precisar do corpus."""
    def __init__(self, vocab: dict):
        self.chars       = vocab["chars"]
        self.vocab_size  = len(self.chars)
        self.char_to_idx = vocab["char_to_idx"]
        self.idx_to_char = {int(k): ch for k, ch in vocab["idx_to_char"].items()}

    def encode(self, text: str) -> list:
        return [self.char_to_idx[c] for c in text if c in self.char_to_idx]

    def decode(self, ids) -> str:
        return "".join(self.idx_to_char[i] for i in ids)

    def __len__(self):
        return self.vocab_size

tokenizer_aula = BundleTokenizer(bundle["vocab"])

# ── Reconstruir modelo com config salva ───────────────────────────────────────
cfg   = bundle["config"]
model_aula = MiniGPT(**cfg, dropout=0.0).to(device)
model_aula.load_state_dict(bundle["model_state"])
model_aula.eval()

# ── Exibir informações do modelo carregado ────────────────────────────────────
tr   = bundle["training"]
meta = bundle["metadata"]
print("=" * 55)
print("  MODELO CARREGADO COM SUCESSO")
print("=" * 55)
print(f"  Corpus       : {meta['corpus']}")
print(f"  Treinado em  : {meta['timestamp']}")
print(f"  Parâmetros   : {tr['n_params']:,}")
print(f"  Melhor época : {tr['best_epoch']}")
print(f"  Melhor loss  : {tr['best_val']:.4f}")
print(f"  Vocab size   : {tokenizer_aula.vocab_size}")
print(f"  Context len  : {cfg['context_len']}")
print("=" * 55)

# ── Gerar texto ───────────────────────────────────────────────────────────────
print("\n--- Exemplo de geração ---\n")
prompts_demo = [
    "Capitu era formosa",
    "Era uma vez um homem",
    "O tempo passou e eu",
]
for prompt in prompts_demo:
    saida = model_aula.generate(
        prompt, tokenizer_aula,
        max_new_tokens=150,
        temperature=0.8,
        top_k=40,
    )
    print(f"Prompt : {prompt!r}")
    print(f"Output : {saida}\n")
    print("-" * 55)


---
## ✅ Módulo 9 — Resumo e Próximos Passos

### O que construímos

| Componente | Descrição | Parâmetros |
|---|---|---|
| `CharTokenizer` | Tokenizador char-level bidirecional | — |
| `TextDataset` | Janela deslizante com teacher forcing | — |
| `CausalSelfAttention` | Multi-head attention mascarada | ~4 × d² |
| `FeedForward` | MLP por posição com GELU + expansão 4× | ~8 × d² |
| `TransformerBlock` | Pre-LN + Attention + FF + Residuais | ~12 × d² |
| `MiniGPT` | Modelo completo com weight tying | ~6.5 M |

### Conexão com o paper original (Vaswani et al., 2017)

| Componente | Paper | MiniGPT |
|---|---|---|
| Multi-Head Attention | ✅ | ✅ |
| Positional Encoding | Senoidal fixo | Aprendido (GPT-style) |
| Feed-Forward 4× | ✅ | ✅ |
| Residual + LayerNorm | Post-LN | **Pre-LN** (mais estável) |
| Encoder | ✅ | ❌ (não necessário) |
| Cross-Attention | ✅ | ❌ (decoder-only) |

### 🧪 Experimentos Sugeridos

1. **Temperatura**: Experimente τ ∈ {0.3, 0.7, 1.0, 1.5} — como a coerência muda?
2. **Escalar o modelo**: dobre `D_MODEL` e `N_LAYERS` — melhora a qualidade?
3. **Top-p (nucleus sampling)**: implemente como alternativa ao top-k
4. **Sinusoidal PE**: substitua o PE aprendido pelo senoidal do paper original
5. **Corpus maior**: adicione *Memórias Póstumas* ou *Quincas Borba* — o modelo generaliza?
6. **KV Cache**: implemente caching da atenção para acelerar a geração (já vimos em aula!)

### 📚 Leituras Recomendadas

- Vaswani et al. (2017) — *Attention Is All You Need*
- Brown et al. (2020) — *Language Models are Few-Shot Learners* (GPT-3)
- Radford et al. (2019) — *Language Models are Unsupervised Multitask Learners* (GPT-2)
- Karpathy, A. — [*nanoGPT*](https://github.com/karpathy/nanoGPT) (inspiração direta deste notebook)
- Warfield, D. — *Intuitively and Exhaustively Explained* (série no Medium)

---
*Notebook desenvolvido para fins didáticos · Baseado em Dom Casmurro (Machado de Assis, 1899)*
